In [2]:
!nvidia-smi

Mon Nov  2 08:03:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
%cd /kaggle/working
!git clone https://github.com/shivsky-108/SOP-Track-Quantization-.git
%cd SOP-Track-Quantization-
!pip install -q -U "transformers>=4.45" datasets accelerate pillow bitsandbytes
!nvidia-smi > logs/nvidia_smi.txt
!pip freeze > logs/pip_freeze.txt

/kaggle/working
fatal: destination path 'SOP-Track-Quantization-' already exists and is not an empty directory.
/kaggle/working/SOP-Track-Quantization-


In [16]:
!python baseline_fp16.py --n_eval 50 2>&1 | tee logs/baseline_smoke.log

Generating test split: 100%|██████████| 2017/2017 [00:00<00:00, 9934.00 examples/s]
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
{
  "name": "baseline_fp16",
  "seed": 40,
  "model_size_gb": 4.417971872,
  "acc": 82.0,
  "n": 50,
  "prefill_sec_per_sample": 0.1968439817428589,
  "peak_mem_gb": 4.674500096,
  "nan_count": 0
}


In [27]:
%cd /kaggle/working/SOP-Track-Quantization-
!git pull origin main
!ls

/kaggle/working/SOP-Track-Quantization-
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 401 bytes | 401.00 KiB/s, done.
From https://github.com/shivsky-108/SOP-Track-Quantization-
 * branch            main       -> FETCH_HEAD
   8fd13c9..c59d9a4  main       -> origin/main
Updating 594aa98..c59d9a4
Fast-forward
 bnb_quant.py | 9 +++++----
 1 file changed, 5 insertions(+), 4 deletions(-)
baseline_fp16.py  common.py  mquant_sim.py  README.md	      summarize.py
bnb_quant.py	  logs	     __pycache__    requirements.txt


In [18]:
!rm -f logs/results.jsonl

In [19]:
!python baseline_fp16.py 2>&1 | tee logs/baseline_fp16.log

Loading weights: 100%|██████████| 729/729 [00:01<00:00, 374.02it/s]
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
{
  "name": "baseline_fp16",
  "seed": 40,
  "model_size_gb": 4.417971872,
  "acc": 80.4,
  "n": 500,
  "prefill_sec_per_sample": 0.15217690134048462,
  "peak_mem_gb": 4.6825856,
  "nan_count": 0
}


In [23]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")

!git config user.name "shivsky-108"
!git config user.email "f20230640@pilani.bits-pilani.ac.in"
!git add -A
!git commit -m "FP16 baseline results and logs"
!git push https://{token}@github.com/shivsky-108/SOP-Track-Quantization-.git main

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 4 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 9.93 KiB | 4.96 MiB/s, done.
Total 8 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 1 local object.
To https://github.com/shivsky-108/SOP-Track-Quantization-.git
   8fd13c9..594aa98  main -> main


In [24]:
!cat logs/results.jsonl

{"name": "baseline_fp16", "seed": 40, "model_size_gb": 4.417971872, "acc": 80.4, "n": 500, "prefill_sec_per_sample": 0.15217690134048462, "peak_mem_gb": 4.6825856, "nan_count": 0}


In [28]:
!python bnb_quant.py --bits 8 2>&1 | tee logs/bnb8.log

Loading weights: 100%|██████████| 729/729 [00:04<00:00, 165.87it/s]
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
{
  "name": "bnb_8bit",
  "seed": 40,
  "model_size_gb": 2.44455184,
  "acc": 81.0,
  "n": 500,
  "prefill_sec_per_sample": 0.3721170792579651,
  "peak_mem_gb": 2.717242368,
  "nan_count": 0
}


In [29]:
!python bnb_quant.py --bits 4 2>&1 | tee logs/bnb4.log

Loading weights: 100%|██████████| 729/729 [00:02<00:00, 249.26it/s]
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
{
  "name": "bnb_4bit",
  "seed": 40,
  "model_size_gb": 1.457841824,
  "acc": 78.2,
  "n": 500,
  "prefill_sec_per_sample": 0.1895974884033203,
  "peak_mem_gb": 1.87951872,
  "nan_count": 0
}


In [30]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")

!git config user.name "shivsky-108"
!git config user.email "f20230640@pilani.bits-pilani.ac.in"
!git add -A
!git commit -m "bitsandbytes 8-bit and 4-bit results"
!git push https://{token}@github.com/shivsky-108/SOP-Track-Quantization-.git main

[main f02b2f9] bitsandbytes 8-bit and 4-bit results
 3 files changed, 30 insertions(+)
 create mode 100644 logs/bnb4.log
 create mode 100644 logs/bnb8.log
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 4 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.17 KiB | 2.17 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 2 local objects.
To https://github.com/shivsky-108/SOP-Track-Quantization-.git
   c59d9a4..f02b2f9  main -> main


In [31]:
!cat logs/results.jsonl

{"name": "baseline_fp16", "seed": 40, "model_size_gb": 4.417971872, "acc": 80.4, "n": 500, "prefill_sec_per_sample": 0.15217690134048462, "peak_mem_gb": 4.6825856, "nan_count": 0}
{"name": "bnb_8bit", "seed": 40, "model_size_gb": 2.44455184, "acc": 81.0, "n": 500, "prefill_sec_per_sample": 0.3721170792579651, "peak_mem_gb": 2.717242368, "nan_count": 0}
{"name": "bnb_4bit", "seed": 40, "model_size_gb": 1.457841824, "acc": 78.2, "n": 500, "prefill_sec_per_sample": 0.1895974884033203, "peak_mem_gb": 1.87951872, "nan_count": 0}


In [32]:
!python mquant_sim.py --n_eval 50 --n_calib 8 2>&1 | tee logs/sim_smoke.log

Loading weights: 100%|██████████| 729/729 [00:02<00:00, 362.19it/s]
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
Replaced 196 linear layers
visual tokens in 1st sample: 234 of 344
visual/text abs-max ratio: mean=0.89 max=2.24  layers with ratio>1: 78/196
{
  "name": "sim_W8A8_fp",
  "seed": 40,
  "ratio_mean": 0.892259393704637,
  "ratio_max": 2.2447001934051514,
  "acc": 82.0,
  "n": 50,
  "prefill_sec_per_sample": 0.15749250888824462,
  "peak_mem_gb": 7.313998336,
  "nan_count": 0
}
{
  "name": "sim_W8A8_dyn_token",
  "seed": 40,
  "ratio_mean": 0.892259393704637,
  "ratio_max": 2.2447001934051514,
  "acc": 82.0,
  "n": 50,
  "prefill_sec_per_sample": 0.18936401844024658,
  "peak_mem_gb": 7.313998336,
  "nan_

In [33]:
#to clear smoke results from log files
!cp logs/results.jsonl logs/results_before_sim.jsonl

In [34]:
import json
rows = [json.loads(l) for l in open("logs/results.jsonl")]
keep = [r for r in rows if r["n"] == 500]
open("logs/results.jsonl", "w").write("".join(json.dumps(r) + "\n" for r in keep))
print(len(keep), "rows kept")   # should print 3

3 rows kept


In [37]:
!python mquant_sim.py 2>&1 | tee logs/sim_w8a8.log

Loading weights: 100%|██████████| 729/729 [00:01<00:00, 382.24it/s]
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
Replaced 196 linear layers
visual tokens in 1st sample: 234 of 344
visual/text abs-max ratio: mean=0.88 max=1.66  layers with ratio>1: 77/196
{
  "name": "sim_W8A8_fp",
  "seed": 40,
  "ratio_mean": 0.8759509662774448,
  "ratio_max": 1.6637167930603027,
  "acc": 80.4,
  "n": 500,
  "prefill_sec_per_sample": 0.15800279188156127,
  "peak_mem_gb": 7.32110592,
  "nan_count": 0
}
{
  "name": "sim_W8A8_dyn_token",
  "seed": 40,
  "ratio_mean": 0.8759509662774448,
  "ratio_max": 1.6637167930603027,
  "acc": 81.0,
  "n": 500,
  "prefill_sec_per_sample": 0.18607265853881835,
  "peak_mem_gb": 7.32110592,
  "na

In [38]:
import json
rows = [json.loads(l) for l in open("logs/results.jsonl")]
keep = [r for r in rows if r["n"] == 500]
open("logs/results.jsonl", "w").write("".join(json.dumps(r) + "\n" for r in keep))
print(len(keep), "rows kept")   # should print 7

7 rows kept


In [39]:
!python summarize.py | tee logs/results_table.md

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
!git config user.name "shivsky-108"
!git config user.email "f20230640@pilani.bits-pilani.ac.in"
!git add -A
!git commit -m "Simulated W8A8 full run"
!git push https://{token}@github.com/shivsky-108/SOP-Track-Quantization-.git main

| Config | Acc (%) | Peak mem (GB) | Prefill s/sample | Model size (GB) |
|---|---|---|---|---|
| baseline_fp16 | 80.40 | 4.68 | 0.152 | 4.42 |
| bnb_8bit | 81.00 | 2.72 | 0.372 | 2.44 |
| bnb_4bit | 78.20 | 1.88 | 0.190 | 1.46 |
| sim_W8A8_fp | 80.40 | 7.32 | 0.158 | nan |
| sim_W8A8_dyn_token | 81.00 | 7.32 | 0.186 | nan |
| sim_W8A8_static_shared | 64.60 | 7.32 | 0.179 | nan |
| sim_W8A8_static_msq | 64.40 | 7.32 | 0.181 | nan |
[main 3ada373] Simulated W8A8 full run
 3 files changed, 56 insertions(+), 1 deletion(-)
Enumerating objects: 16, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 4 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (13/13), 3.33 KiB | 3.33 MiB/s, done.
Total 13 (delta 8), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (8/8), completed with 2 local objects.
To https://github.com/shivsky-108/SOP-Track-Quantization-.git
   f02b2f9..3ada373  main -> main
